<a href="https://colab.research.google.com/github/Hanzet22/GAN-Furry-Art-Photo/blob/main/GAN%20Furry%20V4.2.1%20(Art%20And%20Photo%20(Maintenance)).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HyperGaruda Real-ESRGAN — Fixed Version

**Model tersedia:**
- `anime` — Ilustrasi 2D / Furry Art
- `photo` — Foto Fursuit / Real Photo
- `sharpen-effect` — Foto Fursuit Yang Dipertajam dan Diperkuat strukturnya (Pengurangan noise kurang)
- `NKMD` — Foto Fursuit / Real, mempertajam edges dan menghapus efek artefak
- `ultrasharp` — Foto Fursuit / Real, menghapus artefak, upscale hingga 4X (coming soon X8)
- `foolhardy-remacri` — Foto Fursuit / Real, peningkatan detail ekstrem hingga pori-pori fur

Pilih model di cell berikutnya sebelum run.

---

# HyperGaruda Real-ESRGAN — Fixed Version

**Available models:**
- `anime` — 2D illustrations / Furry art
- `photo` — Fursuit photos / Real photos
- `sharpen-effect` — A fursuit photo that has been sharpened and has its structure enhanced (Less noise reduction)
- `NKMD` — Fursuit / Real photos, sharpens edges and removes compression artifacts
- `ultrasharp` — Fursuit / Real photos, removes artifacts, upscale up to 4X (X8 coming soon)
- `foolhardy-remacri` — Fursuit / Real photos, extreme detail enhancement up to fur pores

Select a model in the next cell before running.

In [3]:
MODE = 'ultrasharp'  # ganti ke 'photo' kalau mau upscale foto fursuit

# ==========================================
# CONFIG — PILIH MODEL DI SINI
# ==========================================
# 'anime' = ilustrasi 2D, furry art, digital art
# 'photo' = foto fursuit, foto real, IRL photo
# 'sharpen-effect' = foto fursuit, foto real, sharpen dan struktur diperkuat, Pengurangan Noise kurang
# 'NKMD' = foto fursuit, real, sharpen dan struktur diperuat, mempertajam edges, penghapusan efek artefak
# 'ultrasharp' = foto fursuit, real, sharpen dan struktur diperkuat, menghapus artefak, upscale hingga 4X (coming soon X8)
# 'foolhardy-remacri' = foto fursuit, real, peningkatan detail hingga pori pori fur

# --- GROUP 1: Real-ESRGAN Models ---
if MODE == 'anime':
    MODEL_NAME = 'RealESRGAN_x4plus_anime_6B'
    MODEL_URL  = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth'
    MODEL_FILE = 'weights/RealESRGAN_x4plus_anime_6B.pth'
elif MODE == 'photo':
    MODEL_NAME = 'RealESRGAN_x4plus'
    MODEL_URL  = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    MODEL_FILE = 'weights/RealESRGAN_x4plus.pth'
elif MODE == 'sharpen-effect':
    MODEL_NAME = 'RealESRNet_x4plus'
    MODEL_URL  = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth'
    MODEL_FILE = 'weights/RealESRNet_x4plus.pth'

# --- GROUP 2: HuggingFace Models ---
elif MODE == 'NKMD':
    MODEL_NAME = 'NKMD'
    MODEL_URL  = 'https://huggingface.co/art0123/Models_collection/resolve/main/upscale_models/4x_NMKD-Superscale-SP_178000_G.pth'
    MODEL_FILE = 'weights/4x_NMKD-Superscale-SP_178000_G.pth'
elif MODE == 'ultrasharp':
    MODEL_NAME = 'Ultrasharp'
    MODEL_URL  = 'https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth'
    MODEL_FILE = 'weights/4x-Ultrasharp.pth'
elif MODE == 'foolhardy-remacri':
    MODEL_NAME = 'Foolhardy Remacri'
    MODEL_URL  = 'https://huggingface.co/FacehugmanIII/4x_foolhardy_Remacri/resolve/main/4x_Foolhardy_Remacri.pth'
    MODEL_FILE = 'weights/4x_Foolhardy_Remacri.pth'
else:
    raise ValueError(f'MODE "{MODE}" tidak valid. Pilih dari model yang tersedia di atas.')

print(f'✅ Mode: {MODE.upper()} | Model: {MODEL_NAME}')

✅ Mode: ULTRASHARP | Model: Ultrasharp


In [18]:
import os
import site

# ==========================================
# 1. CLONE REPO & DIRECTORY SETUP
# ==========================================
print('📥 Checking Real-ESRGAN Repository...')
if not os.path.exists('Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git

# Always ensure we are inside the directory for the following steps
if 'Real-ESRGAN' in os.getcwd():
    print('✅ Already in Real-ESRGAN folder.')
else:
    %cd Real-ESRGAN
    print('✅ Entered Real-ESRGAN folder.')

# ==========================================
# 2. INSTALL DEPENDENCIES
# ==========================================
print('🔧 Installing dependencies (this may take a minute)...')
!pip install -q basicsr facexlib gfpgan
!pip install -q -r requirements.txt
!python setup.py develop -q
print('✅ Dependencies installed.')

# ==========================================
# 3. AUTOMATIC MODEL DOWNLOAD (DYNAMIC)
# ==========================================
# Ensure weights folder exists inside Real-ESRGAN
os.makedirs('weights', exist_ok=True)

# The MODEL_FILE and MODEL_URL come from the selection cell
if not os.path.exists(MODEL_FILE):
    print(f"📥 Downloading model: {MODEL_NAME}...")
    !wget -q -O {MODEL_FILE} {MODEL_URL}
    print("✅ Download complete!")
else:
    print(f"📦 Model {MODEL_NAME} is already present.")

print(f'🚀 Active Mode: {MODE.upper()}')

# ==========================================
# DOWNLOAD TENSORFLOW VERSI 2.12.0 (KOMPATIBEL)
# ==========================================
!pip uninstall -y tensorflow
!pip install -q tensorflow==2.12.0
print('✅ TensorFlow 2.12.0 installed!')

# ==========================================
# 4. FIX BASICSR COMPATIBILITY
# ==========================================
print('🔧 Patching basicsr compatibility...')
try:
    import basicsr
    basicsr_path = os.path.dirname(basicsr.__file__)
    files_to_patch = [
        os.path.join(basicsr_path, 'data/degradations.py'),
        os.path.join(basicsr_path, 'utils/img_process_util.py'),
    ]

    for fpath in files_to_patch:
        if os.path.exists(fpath):
            with open(fpath, 'r') as f:
                content = f.read()
            patched = content.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            )
            with open(fpath, 'w') as f:
                f.write(patched)
            print(f'✅ Patched: {os.path.basename(fpath)}')
except Exception as e:
    print(f'⚠️ Patching failed: {e}')

print('✨ Setup Finished!')

📥 Checking Real-ESRGAN Repository...
Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Total 759 (delta 0), reused 0 (delta 0), pack-reused 759 (from 1)
Receiving objects: 100% (759/759), 5.39 MiB | 21.80 MiB/s, done.
Resolving deltas: 100% (408/408), done.
✅ Already in Real-ESRGAN folder.
🔧 Installing dependencies (this may take a minute)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 94.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openvino-dev 2024.6.0 requires numpy<2.0.0,>=1.16.6, but you have numpy 2.0.2 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 3.1 which is incompatible.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 3.1 which is incompatible.
mapclassify 2.10.0 r

In [5]:
# ==========================================
# CLONE CUSTOM-INFERENCE-UNIVERSAL
# ==========================================
print('📥 Cloning Custom-Inference-Universal repository...')

# Buat folder baru (opsional, kalau mau rapi)
!mkdir -p inference_V2

# Masuk ke folder
%cd inference_V2

# Clone repo (pastikan URL benar)
!git clone https://github.com/Hanzet22/Custom-Inference-Universal.git .

# Kembali ke folder sebelumnya
%cd ..

print('✅ Repository cloned successfully!')

# ==========================================
# CEK FILE YANG SUDAH ADA
# ==========================================
import os

files = os.listdir('inference_V2')
print(f'📂 Files in inference_V2: {len(files)}')
for f in sorted(files):
    print(f'  - {f}')

# ==========================================
# INSTALL DEPENDENCIES DARI REPO
# ==========================================
%cd inference_V2

if os.path.exists('Requirements.txt'):
    print('🔧 Installing requirements...')
    !pip install -q -r Requirements.txt
    print('✅ Requirements installed!')
else:
    print('⚠️ requirements.txt not found in repo.')

%cd ..

# ==========================================
# FIX detector.py (kalau belum)
# ==========================================
!sed -i '1i from pathlib import Path' inference_V2/detector.py
print('✅ detector.py fixed (Path import added)')

📥 Cloning Custom-Inference-Universal repository...
/content/Real-ESRGAN/inference_V2
Cloning into '.'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 121 (delta 45), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 50.88 KiB | 5.09 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/Real-ESRGAN
✅ Repository cloned successfully!
📂 Files in inference_V2: 12
  - .git
  - LICENSE
  - README.md
  - Requirements.txt
  - architectures.py
  - config.py
  - detector.py
  - loader.py
  - main.py
  - telegram_notifier.py
  - upscaler.py
  - utils.py
/content/Real-ESRGAN/inference_V2
🔧 Installing requirements...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × pip subprocess to install 

In [13]:
# ==========================================
# DETEKSI & INSTALL PACKAGE (HANYA YANG BELUM)
# ==========================================
import subprocess
import sys

%cd inference_V2

# Baca Requirements.txt
with open('Requirements.txt', 'r') as f:
    packages = [line.strip() for line in f if line.strip() and not line.startswith('#')]

# Cek package mana yang sudah terinstall
installed = []
missing = []

for pkg in packages:
    # Pisahkan nama package dari versi (kalau ada)
    pkg_name = pkg.split('==')[0].split('>=')[0].split('<=')[0].split('[')[0].strip()

    # Cek apakah sudah terinstall
    try:
        __import__(pkg_name)
        installed.append(pkg)
        print(f'✅ {pkg} sudah terinstall')
    except ImportError:
        missing.append(pkg)
        print(f'⚠️ {pkg} belum terinstall')

print('=' * 50)
print(f'📊 Package sudah terinstall: {len(installed)}')
print(f'📦 Package belum terinstall: {len(missing)}')

# Install yang belum
if missing:
    print('🔧 Menginstall package yang belum...')
    for pkg in missing:
        !pip install -q {pkg}
    print('✅ Semua package berhasil diinstall!')
else:
    print('✅ Semua package sudah terinstall!')

%cd ..

/content/Real-ESRGAN/inference_V2
⚠️ basicsr belum terinstall
✅ facexlib sudah terinstall
⚠️ gfpgan belum terinstall
⚠️ opencv-python belum terinstall
✅ torch sudah terinstall
✅ torchvision sudah terinstall
⚠️ onnxruntime belum terinstall
✅ safetensors sudah terinstall
✅ tensorflow sudah terinstall
✅ requests sudah terinstall
✅ numpy sudah terinstall
⚠️ Pillow belum terinstall
✅ tqdm sudah terinstall
⚠️ realesrgan belum terinstall
✅ scipy sudah terinstall
⚠️ pyyaml belum terinstall
✅ lmdb sudah terinstall
✅ einops sudah terinstall
⚠️ onnx belum terinstall
⚠️ opencv-contrib-python belum terinstall
✅ numba sudah terinstall
⚠️ boto3 belum terinstall
✅ h5py sudah terinstall
⚠️ coremltools belum terinstall
⚠️ openvino-dev belum terinstall
⚠️ tensorrt belum terinstall
⚠️ protobuf belum terinstall
✅ psutil sudah terinstall
✅ packaging sudah terinstall
✅ wheel sudah terinstall
✅ setuptools sudah terinstall
✅ cython sudah terinstall
⚠️ ffmpeg-python belum terinstall
✅ imageio sudah terinstall
⚠

ModuleNotFoundError: No module named 'torchvision.transforms.functional_tensor'

In [14]:
# Menghapus semua file di dalam folder inputs tanpa menghapus foldernya
!rm -rf inputs/*

In [15]:
# ==========================================
# 5. UPLOAD GAMBAR
# ==========================================
from google.colab import files
import glob

os.makedirs('inputs', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f'📂 Upload gambar ({MODE} mode)...')
uploaded = files.upload()

for filename in uploaded.keys():
    new_name = filename.replace(' ', '_')
    !mv "{filename}" "inputs/{new_name}"
    print(f'   ➕ {new_name}')

📂 Upload gambar (ultrasharp mode)...


Saving 20260624_123937.jpg to 20260624_123937.jpg
   ➕ 20260624_123937.jpg


In [16]:

# ==========================================
# 6. UPSCALE (V2 - FIXED)
# ==========================================
print(f'🚀 Upscaling ({MODE.upper()} mode | {MODEL_NAME})...')

!python /content/inference_V2/main.py \
    --mode upscale \
    --model "{MODEL_FILE}" \
    --input inputs \
    --output results \
    --scale 4 \
    --tile 512 \
    --tile_pad 32 \
    --ext png

🚀 Upscaling (ULTRASHARP mode | Ultrasharp)...
scikit-learn version 1.6.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
XGBoost version 3.2.0 has not been tested with coremltools. You may run into unexpected errors. XGBoost 1.4.2 is the most recent version that has been tested.
TensorFlow version 2.20.0 has not been tested with coremltools. You may run into unexpected errors. TensorFlow 2.12.0 is the most recent version that has been tested.
Torch version 2.11.0+cu128 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.
Traceback (most recent call last):
  File "/content/inference_V2/main.py", line 7, in <module>
    from loader import load_model_universal
  File "/content/inference_V2/loader.py", line 9, in <module>
    from architectures import detect_architecture_from_state_dict
  File "/content/inference_V2/architectures.py", l

In [9]:
# ==========================================
# 7. DOWNLOAD HASIL
# ==========================================
from google.colab import files
import glob

print('\n🎉 Done! Preparing download...')

if os.path.exists('results') and os.listdir('results'):
    result_files = glob.glob('results/*.png')
    print(f'✅ {len(result_files)} image(s) upscaled.')
    !zip -r /content/results.zip results
    files.download('/content/results.zip')
    print('📥 Download started.')
else:
    print('⚠️ No results found. Check error logs above.')


🎉 Done! Preparing download...
⚠️ No results found. Check error logs above.


In [10]:
# run ini sebelum memulai

import time

print("🚀 Script Anti-Idle Aktif! Tekan tombol STOP jika ingin menyudahi.")
try:
    while True:
        # Mencetak timestamp setiap 60 detik agar log terus berjalan
        print(f"⏳ Server Masih Hidup: {time.strftime('%H:%M:%S')}")
        time.sleep(60)
except KeyboardInterrupt:
    print("⏹️ Script Anti-Idle dihentikan manual.")

🚀 Script Anti-Idle Aktif! Tekan tombol STOP jika ingin menyudahi.
⏳ Server Masih Hidup: 06:22:27
⏹️ Script Anti-Idle dihentikan manual.
